---

<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">NLP: pré-processamento e embeddings</h1>

<h3 align="center">Ângelo Rosa</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
</div>

---

## 1. Contexto e objetivo

Preparar o texto dos reviews para o classificador de sentimento do módulo 2. Este notebook entrega
dois artefatos em `data/embeddings/`:

1. `emb.npy`: um vetor denso por review, gerado por um modelo de linguagem pré-treinado
   (`utils.embedding`, trocável pelo `EMBEDDING_MODEL` do `config.py`).
2. `meta.parquet`: `review_id`, nota, categoria, o rótulo de sentimento e o texto já tratado.

O rótulo vem da nota (1-2 negativo, 3 neutro, 4-5 positivo). É um rótulo fraco, mas é o único sinal
disponível para todos os reviews. Os mesmos embeddings servem ao módulo 3 (busca semântica).

In [1]:
import html
import os
import re
import sqlite3

import numpy as np
import pandas as pd

from config import DB_PATH, EMBEDDING_DIR, EMBED_MAX_CHARS, SENTIMENT_LABELS
from utils.embedding import embed
from utils.sql import query_sql

os.makedirs(EMBEDDING_DIR, exist_ok=True)
EMB_PATH = f"{EMBEDDING_DIR}/emb.npy"
IDS_PATH = f"{EMBEDDING_DIR}/review_ids.npy"
META_PATH = f"{EMBEDDING_DIR}/meta.parquet"

c:\Users\PC\Documents\Projetos\amazon-reviews-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carrega os reviews

Só o necessário: identificador, título, texto, nota e categoria, na ordem do `review_id` (essa ordem
é o que alinha a matriz de embeddings à tabela de metadados).

In [2]:
conn = sqlite3.connect(DB_PATH)
reviews = query_sql(conn, """
SELECT review_id, title, text, rating
FROM reviews ORDER BY review_id
""")
conn.close()

reviews["sentiment_label"] = np.where(reviews["rating"] <= 2, 0,
                                      np.where(reviews["rating"] == 3, 1, 2))
print(f"{len(reviews):,} reviews carregados")
reviews["sentiment_label"].map(SENTIMENT_LABELS).value_counts()

111,851 reviews carregados


sentiment_label
positivo    72117
negativo    30080
neutro       9654
Name: count, dtype: int64

## 3. Tratamento do texto

O que o modelo recebe é `título + texto`, porque o título costuma carregar a opinião de forma direta
("Broken clasps", "Love these!"). O tratamento é deliberadamente leve: remover HTML (cerca de 9% dos
reviews trazem `<br />`), resolver entidades como `&amp;`, normalizar espaços e truncar em
`EMBED_MAX_CHARS`.

Nada de minúsculas, remoção de pontuação ou stopwords: o transformer usa justamente a ordem, a
pontuação e o contexto que esse tipo de limpeza destrói.

In [3]:
TAGS_HTML = re.compile(r"<[^>]+>")


def tratar(titulo: str, texto: str) -> str:
    """Junta título e texto, remove HTML e espaços repetidos, e trunca."""
    bruto = html.unescape(f"{titulo}. {texto}")
    return " ".join(TAGS_HTML.sub(" ", bruto).split())[:EMBED_MAX_CHARS]


reviews["text_prep"] = [tratar(t, x) for t, x in zip(reviews["title"], reviews["text"])]

sujos = reviews[reviews["text"].str.contains(r"<[^>]+>|&[a-z]+;", case=False, regex=True)]
exemplo = sujos.loc[sujos["text"].str.len().idxmin()]
print(f"reviews com HTML ou entidades: {len(sujos):,}")
print("vazios após o tratamento:", int((reviews["text_prep"].str.len() == 0).sum()), "\n")
print("ANTES :", repr(f'{exemplo["title"]}. {exemplo["text"]}'))
print("DEPOIS:", repr(tratar(exemplo["title"], exemplo["text"])))

reviews com HTML ou entidades: 8,076
vazios após o tratamento: 0 

ANTES : 'Good stuff.. Great product<br />!'
DEPOIS: 'Good stuff.. Great product !'


### Por que o tratamento para aqui

É tentador acrescentar lematização e remoção de stopwords, que é o pré-processamento clássico de
NLP. Aqui isso **pioraria** o resultado, por três razões:

- **A negação sumiria.** As listas de stopwords do NLTK e do scikit-learn incluem `no`, `not` e
  `never`. Remover `not` transforma "does not work" em "work". A análise de termos do notebook 02
  mostra que os bigramas de negação são justamente o que separa as classes.
- **A lematização briga com o tokenizador.** O MiniLM usa WordPiece e foi treinado com as formas
  flexionadas: "loved" carrega intensidade que "love" não carrega.
- **Sem stopwords, a sintaxe quebra.** O mecanismo de atenção foi treinado em texto natural; frases
  sem artigos e preposições ficam fora da distribuição de treino.

A regra prática: limpar ruído (HTML, entidades, espaços) é pré-processamento legítimo para um
modelo de linguagem; normalizar a língua não é. O NLTK entra no notebook 02, para analisar
vocabulário, e no 04, como linha de base sem treino, nunca antes do `embed()`.

## 4. Embeddings densos

`utils.embedding.embed` transforma texto em vetor com o SentenceTransformer configurado. Os vetores
saem normalizados, então o produto interno entre dois deles já é a similaridade de cosseno: vai de
-1 (sentidos opostos) a 1 (mesma direção), e quanto mais perto de 1, mais próximos os significados.

Antes de rodar na base inteira, o exemplo abaixo mostra a mecânica em duas frases.

In [5]:
# duas reclamações do mesmo problema, escritas sem nenhuma palavra em comum
frase_a = "The battery died after two days."
frase_b = "It stopped holding a charge almost immediately."

vetor_a, vetor_b = embed([frase_a, frase_b], show_progress=False)

print("cada frase virou um vetor de", vetor_a.shape[0], "números")
print("início do vetor A:", vetor_a[:5].round(3))
print("similaridade A·B :", round(float(vetor_a @ vetor_b), 3))

c:\Users\PC\Documents\Projetos\amazon-reviews-chatbot\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\PC\Documents\Projetos\amazon-reviews-chatbot\.venv\Lib\site-pack

cada frase virou um vetor de 384 números
início do vetor A: [-0.005  0.04  -0.006  0.054  0.048]
similaridade A·B : 0.59


As duas frases não têm uma palavra sequer em comum ("battery died" contra "stopped holding a
charge") e mesmo assim ficam próximas, com similaridade em torno de 0,59: o embedding aproxima por
sentido, não por vocabulário. Um modelo de saco de palavras daria zero aqui. É isso que permite ao
classificador do próximo notebook reconhecer uma reclamação escrita de um jeito que ele nunca viu
no treino.

Agora o mesmo para a base inteira. Se o cache já existir e cobrir exatamente os mesmos `review_id`,
apenas recarregamos: re-executar o notebook não paga o custo de embutir tudo de novo.

In [6]:
textos = reviews["text_prep"].tolist()
ids = reviews["review_id"].to_numpy()

cache_valido = (os.path.exists(EMB_PATH) and os.path.exists(IDS_PATH)
                and np.array_equal(np.load(IDS_PATH), ids))
if cache_valido:
    emb = np.load(EMB_PATH)
    print("cache válido, recarregando:", emb.shape)
else:
    emb = embed(textos)
    np.save(EMB_PATH, emb)
    np.save(IDS_PATH, ids)
    print("embeddings gerados:", emb.shape)

reviews[["review_id", "rating", "sentiment_label", "text_prep"]].to_parquet(META_PATH)
print("artefatos em", EMBEDDING_DIR, "| dimensão:", emb.shape[1])

Batches: 100%|██████████| 1748/1748 [09:01<00:00,  3.23it/s]


embeddings gerados: (111851, 384)
artefatos em data/embeddings | dimensão: 384


## 5. Sanidade: os vizinhos fazem sentido?

Teste rápido de qualidade: dado um review, recuperar os mais parecidos por cosseno. Se os embeddings
capturam significado, os vizinhos falam do mesmo assunto, mesmo usando outras palavras.

In [7]:
def vizinhos(i: int, k: int = 1) -> pd.DataFrame:
    """Os k reviews mais parecidos com o review da posição i."""
    sims = emb @ emb[i]
    top = np.argsort(-sims)[1 : k + 1]
    return pd.DataFrame({
        "similaridade": sims[top].round(2),
        "nota": reviews["rating"].to_numpy()[top].astype(int),
        "review": [textos[j][:70] for j in top],
    })


print("CONSULTA:", textos[100][:110])
vizinhos(1000)

CONSULTA: Pleased. This is an amazing product. And a great way to try out the other versions.


,similaridade,nota,review
0,0.87,5,Good Value. Nice lashes for everyday wear or a...


## 6. Encerramento

Gravado em `data/embeddings/`:

- `emb.npy`: matriz de embeddings (float32, uma linha por review).
- `review_ids.npy`: a ordem dos `review_id` que indexa a matriz.
- `meta.parquet`: `review_id`, `rating`, `category`, `sentiment_label` e `text_prep`.

O próximo notebook (`04_m2_sentiment_model`) treina o **XGBoost** sobre esses vetores e grava as
predições no banco.